<a href="https://colab.research.google.com/github/Profgit-01/Credit-Risk-Classification/blob/main/Credit_Risk_Modelling_and_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from scipy.stats import chi2_contingency
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
import warnings
import os
import time

In [11]:
!pip install scikit-learn


In [12]:
!pip install statsmodels


In [13]:
# Load the dataset
a1 = pd.read_excel("/content/Internal_Bank_Dataset.xlsx")
a2 = pd.read_excel("/content/External_Cibil_Dataset.xlsx")

In [14]:
!pip install openpyxl


In [15]:
df1 = a1.copy()
df2 = a2.copy()

In [16]:
df1.T


,0,1,2,3,4,5,6,7,8,9,...,51326,51327,51328,51329,51330,51331,51332,51333,51334,51335
PROSPECTID,1.0,2.0,3.000,4.0,5.000,6.000,7.000,8.000,9.0,10.0,...,51327.000,51328.0,51329.0,51330.0,51331.0,51332.000,51333.00,51334.0,51335.0,51336.0
Total_TL,5.0,1.0,8.000,1.0,3.000,6.000,3.000,6.000,1.0,2.0,...,9.000,1.0,1.0,4.0,30.0,3.000,4.00,2.0,2.0,1.0
Tot_Closed_TL,4.0,0.0,0.000,0.0,2.000,5.000,1.000,4.000,0.0,1.0,...,4.000,1.0,0.0,4.0,21.0,0.000,2.00,1.0,1.0,0.0
Tot_Active_TL,1.0,1.0,8.000,1.0,1.000,1.000,2.000,2.000,1.0,1.0,...,5.000,0.0,1.0,0.0,9.0,3.000,2.00,1.0,1.0,1.0
Total_TL_opened_L6M,0.0,0.0,1.000,1.0,0.000,0.000,1.000,0.000,0.0,0.0,...,1.000,0.0,1.0,0.0,3.0,1.000,0.00,1.0,0.0,0.0
Tot_TL_closed_L6M,0.0,0.0,0.000,0.0,0.000,0.000,1.000,0.000,0.0,0.0,...,2.000,0.0,0.0,0.0,3.0,0.000,1.00,1.0,0.0,0.0
pct_tl_open_L6M,0.0,0.0,0.125,1.0,0.000,0.000,0.333,0.000,0.0,0.0,...,0.111,0.0,1.0,0.0,0.1,0.333,0.00,0.5,0.0,0.0
pct_tl_closed_L6M,0.0,0.0,0.000,0.0,0.000,0.000,0.333,0.000,0.0,0.0,...,0.222,0.0,0.0,0.0,0.1,0.000,0.25,0.5,0.0,0.0
pct_active_tl,0.2,1.0,1.000,1.0,0.333,0.167,0.667,0.333,1.0,0.5,...,0.556,0.0,1.0,0.0,0.3,1.000,0.50,0.5,0.5,1.0
pct_closed_tl,0.8,0.0,0.000,0.0,0.667,0.833,0.333,0.667,0.0,0.5,...,0.444,1.0,0.0,1.0,0.7,0.000,0.50,0.5,0.5,0.0


In [17]:
# Remove nulls
df1 = df1.loc[df1['Age_Oldest_TL'] != -99999]

In [18]:
columns_to_be_removed = []

for i in df2.columns:
    if df2.loc[df2[i] == -99999].shape[0] > 10000:
        columns_to_be_removed .append(i)

In [19]:
df2 = df2.drop(columns_to_be_removed, axis =1)
print(f"We have dropped {columns_to_be_removed}")

We have dropped ['time_since_first_deliquency', 'time_since_recent_deliquency', 'max_delinquency_level', 'max_deliq_6mts', 'max_deliq_12mts', 'CC_utilization', 'PL_utilization', 'max_unsec_exposure_inPct']


In [20]:
for i in df2.columns:
    df2 = df2.loc[ df2[i] != -99999 ]

In [21]:
# Checking common column names
for i in list(df1.columns):
    if i in list(df2.columns):
        print (i)

PROSPECTID


In [22]:
# Merge the two dataframes, inner join so that no nulls are present
df = pd. merge ( df1, df2, how ='inner', left_on = ['PROSPECTID'], right_on = ['PROSPECTID'] )

In [23]:
# check how many columns are categorical
for i in df.columns:
    if df[i].dtype == 'object':
        print(i)

MARITALSTATUS
EDUCATION
GENDER
last_prod_enq2
first_prod_enq2
Approved_Flag


In [24]:
# Chi-square test
for i in ['MARITALSTATUS', 'EDUCATION', 'GENDER', 'last_prod_enq2', 'first_prod_enq2']:
    chi2, pval, _, _ = chi2_contingency(pd.crosstab(df[i], df['Approved_Flag']))
    print(i, '---', pval)

MARITALSTATUS --- 3.578180861038862e-233
EDUCATION --- 2.6942265249737532e-30
GENDER --- 1.907936100186563e-05
last_prod_enq2 --- 0.0
first_prod_enq2 --- 7.84997610555419e-287


## Since all the categorical features have pval <=0.05, we will accept all

In [25]:
# VIF for numerical columns
numeric_columns = []
for i in df.columns:
    if df[i].dtype != 'object' and i not in ['PROSPECTID','Approved_Flag']:
        numeric_columns.append(i)

In [26]:
# VIF sequentially check

vif_data = df[numeric_columns]
total_columns = vif_data.shape[1]
columns_to_be_kept = []
column_index = 0

In [27]:
for i in range (0,total_columns):

    vif_value = variance_inflation_factor(vif_data, column_index)
    print (column_index,'---',vif_value)


    if vif_value <= 6:
        columns_to_be_kept.append( numeric_columns[i] )
        column_index = column_index+1

    else:
        vif_data = vif_data.drop([ numeric_columns[i] ] , axis=1)

/tmp/ipykernel_628/1966489429.py:3: UserWarning: The design matrix is poorly conditioned (condition number=2.59e+16). VIF calculations may be numerically unstable.
  vif_value = variance_inflation_factor(vif_data, column_index)
/usr/local/lib/python3.13/dist-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


0 --- 1000799917193443.5


/tmp/ipykernel_628/1966489429.py:3: UserWarning: The design matrix is poorly conditioned (condition number=6.28e+16). VIF calculations may be numerically unstable.
  vif_value = variance_inflation_factor(vif_data, column_index)
/usr/local/lib/python3.13/dist-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


0 --- 1000799917193443.5


/tmp/ipykernel_628/1966489429.py:3: UserWarning: The design matrix is poorly conditioned (condition number=1.09e+16). VIF calculations may be numerically unstable.
  vif_value = variance_inflation_factor(vif_data, column_index)
/usr/local/lib/python3.13/dist-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


0 --- 11.320180023967925


... (46 outputs left out)

/usr/local/lib/python3.13/dist-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/tmp/ipykernel_628/1966489429.py:3: UserWarning: The design matrix is poorly conditioned (condition number=6.79e+15). VIF calculations may be numerically unstable.
  vif_value = variance_inflation_factor(vif_data, column_index)


16 --- 6.9099587485404435


/usr/local/lib/python3.13/dist-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


16 --- 4.92147572178097
17 --- 1000799917193443.5
17 --- 6.640607830175616
17 --- 1.3159154707874958
18 --- 7.625456477064194
18 --- 1.5813808513856944
19 --- 6.247287793745815
19 --- 13.461183034060262
19 --- 2.1153094307817395
20 --- 1.4995355193868976
21 --- 2.170797459734474
22 --- 2.6194659619472316
23 --- 2.293899351764995
24 --- 7.358623292167092
24 --- 2.159662759661637
25 --- 2.866271116689635
26 --- 6.456924734738174
26 --- 2.847110953867859
27 --- 4.498908692151805
28 --- 9.68260534125413
28 --- 5.9672083134871245
29 --- 8.495827332704197
29 --- 5.152084361611115
30 --- 7.280751669212333
30 --- 11.481328638663035
30 --- 2.9675076428960745
31 --- 1.5950059903466
32 --- 12.53092282701146
32 --- 7.8363558324273255
32 --- 3.066874602451879
33 --- 1.4041913778107644
34 --- 1.0484933703163541
35 --- 1.2675679133867654
36 --- 3.127062069210971
37 --- 4.340308450460009
38 --- 1.0008022071023939
39 --- 2.8456129815468905
40 --- 2.2794120029684897
41 --- 15.82733122386849
41 --- 16.38

In [28]:
# check Anova for columns_to_be_kept

from scipy.stats import f_oneway

columns_to_be_kept_numerical = []

for i in columns_to_be_kept:
    a = list(df[i])
    b = list(df['Approved_Flag'])

    group_P1 = [value for value, group in zip(a, b) if group == 'P1']
    group_P2 = [value for value, group in zip(a, b) if group == 'P2']
    group_P3 = [value for value, group in zip(a, b) if group == 'P3']
    group_P4 = [value for value, group in zip(a, b) if group == 'P4']


    f_statistic, p_value = f_oneway(group_P1, group_P2, group_P3, group_P4)

    if p_value <= 0.05:
        columns_to_be_kept_numerical.append(i)

In [29]:
# feature selection is done for cat and num features
# listing all the final features
features = columns_to_be_kept_numerical + ['MARITALSTATUS', 'EDUCATION', 'GENDER', 'last_prod_enq2', 'first_prod_enq2']
df = df[features + ['Approved_Flag']]

In [30]:
# Label encoding for the categorical features
['MARITALSTATUS', 'EDUCATION', 'GENDER' , 'last_prod_enq2' ,'first_prod_enq2']



df['MARITALSTATUS'].unique()
df['EDUCATION'].unique()
df['GENDER'].unique()
df['last_prod_enq2'].unique()
df['first_prod_enq2'].unique()

array(['PL', 'ConsumerLoan', 'others', 'AL', 'HL', 'CC'], dtype=object)

In [31]:
# Ordinal feature -- EDUCATION
# SSC            : 1
# 12TH           : 2
# GRADUATE       : 3
# UNDER GRADUATE : 3
# POST-GRADUATE  : 4
# OTHERS         : 1
# PROFESSIONAL   : 3

# Others has to be verified by the business end user


df.loc[df['EDUCATION'] == 'SSC',['EDUCATION']]              = 1
df.loc[df['EDUCATION'] == '12TH',['EDUCATION']]             = 2
df.loc[df['EDUCATION'] == 'GRADUATE',['EDUCATION']]         = 3
df.loc[df['EDUCATION'] == 'UNDER GRADUATE',['EDUCATION']]   = 3
df.loc[df['EDUCATION'] == 'POST-GRADUATE',['EDUCATION']]    = 4
df.loc[df['EDUCATION'] == 'OTHERS',['EDUCATION']]           = 1
df.loc[df['EDUCATION'] == 'PROFESSIONAL',['EDUCATION']]     = 3

In [32]:
df['EDUCATION'] = df['EDUCATION'].astype(int)

In [33]:
df_encoded = pd.get_dummies(df, columns=['MARITALSTATUS','GENDER', 'last_prod_enq2' ,'first_prod_enq2'])



df_encoded.info()
k = df_encoded.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42064 entries, 0 to 42063
Data columns (total 62 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   pct_tl_open_L6M               42064 non-null  float64
 1   pct_tl_closed_L6M             42064 non-null  float64
 2   Total_TL_opened_L12M          42064 non-null  int64  
 3   Tot_TL_closed_L12M            42064 non-null  int64  
 4   pct_tl_open_L12M              42064 non-null  float64
 5   pct_tl_closed_L12M            42064 non-null  float64
 6   Tot_Missed_Pmnt               42064 non-null  int64  
 7   CC_TL                         42064 non-null  int64  
 8   Home_TL                       42064 non-null  int64  
 9   PL_TL                         42064 non-null  int64  
 10  Secured_TL                    42064 non-null  int64  
 11  Unsecured_TL                  42064 non-null  int64  
 12  Other_TL                      42064 non-null  int64  
 13  A

# Machine Learning model fitting
## Data processing

In [34]:
#Random Forest
y = df_encoded['Approved_Flag']
x = df_encoded. drop ( ['Approved_Flag'], axis = 1 )

In [35]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
rf_classifier = RandomForestClassifier(n_estimators = 200, random_state=42)
rf_classifier.fit(x_train, y_train)
y_pred = rf_classifier.predict(x_test)

In [36]:
accuracy = accuracy_score(y_test, y_pred)
print ()
print(f'Accuracy: {accuracy}')
print ()
precision, recall, f1_score, _ = precision_recall_fscore_support(y_test, y_pred)


for i, v in enumerate(['p1', 'p2', 'p3', 'p4']):
    print(f"Class {v}:")
    print(f"Precision: {precision[i]}")
    print(f"Recall: {recall[i]}")
    print(f"F1 Score: {f1_score[i]}")
    print()


Accuracy: 0.9895399976227267

Class p1:
Precision: 0.9314179796107507
Recall: 0.9911242603550295
F1 Score: 0.9603440038222647

Class p2:
Precision: 0.9986144101346002
Recall: 1.0
F1 Score: 0.9993067247697336

Class p3:
Precision: 0.994413407821229
Recall: 0.940377358490566
F1 Score: 0.9666408068269977

Class p4:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0



In [37]:
pip install xgboost

In [38]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

xgb_classifier = xgb.XGBClassifier(objective='multi:softmax',  num_class=4)

In [39]:
y = df_encoded['Approved_Flag']
x = df_encoded. drop ( ['Approved_Flag'], axis = 1 )

In [40]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [41]:
x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42)

In [42]:
xgb_classifier.fit(x_train, y_train)
y_pred = xgb_classifier.predict(x_test)

In [43]:
accuracy = accuracy_score(y_test, y_pred)
print ()
print(f'Accuracy: {accuracy:.2f}')
print ()

precision, recall, f1_score, _ = precision_recall_fscore_support(y_test, y_pred)

for i, v in enumerate(['p1', 'p2', 'p3', 'p4']):
    print(f"Class {v}:")
    print(f"Precision: {precision[i]}")
    print(f"Recall: {recall[i]}")
    print(f"F1 Score: {f1_score[i]}")
    print()


Accuracy: 0.99

Class p1:
Precision: 0.9868421052631579
Recall: 0.9615384615384616
F1 Score: 0.974025974025974

Class p2:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

Class p3:
Precision: 0.9711324944485566
Recall: 0.990188679245283
F1 Score: 0.9805680119581465

Class p4:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0



In [44]:
from sklearn.tree import DecisionTreeClassifier

In [45]:
y = df_encoded['Approved_Flag']
x = df_encoded. drop ( ['Approved_Flag'], axis = 1 )

In [46]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [47]:
dt_model = DecisionTreeClassifier(max_depth=20, min_samples_split=10)
dt_model.fit(x_train, y_train)
y_pred = dt_model.predict(x_test)

In [48]:
accuracy = accuracy_score(y_test, y_pred)
print ()
print(f"Accuracy: {accuracy:.2f}")
print ()

precision, recall, f1_score, _ = precision_recall_fscore_support(y_test, y_pred)

for i, v in enumerate(['p1', 'p2', 'p3', 'p4']):
    print(f"Class {v}:")
    print(f"Precision: {precision[i]}")
    print(f"Recall: {recall[i]}")
    print(f"F1 Score: {f1_score[i]}")
    print()



Accuracy: 0.99

Class p1:
Precision: 0.9676470588235294
Recall: 0.9733727810650887
F1 Score: 0.9705014749262537

Class p2:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

Class p3:
Precision: 0.979529946929492
Recall: 0.9750943396226415
F1 Score: 0.9773071104387292

Class p4:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0



In [49]:
## as XGboost gave the best results
### We will further finetune it

##Applying standard scaler

from sklearn.preprocessing import StandardScaler

columns_to_be_scaled = ['Age_Oldest_TL','Age_Newest_TL','time_since_recent_payment',
'max_recent_level_of_deliq','recent_level_of_deliq',
'time_since_recent_enq','NETMONTHLYINCOME','Time_With_Curr_Empr']

for i in columns_to_be_scaled:
    column_data = df_encoded[i].values.reshape(-1, 1)
    scaler = StandardScaler()
    scaled_column = scaler.fit_transform(column_data)
    df_encoded[i] = scaled_column


In [50]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

xgb_classifier = xgb.XGBClassifier(objective='multi:softmax',  num_class=4)

In [51]:
y = df_encoded['Approved_Flag']
x = df_encoded. drop ( ['Approved_Flag'], axis = 1 )

In [52]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [53]:
x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42)

In [54]:
xgb_classifier.fit(x_train, y_train)
y_pred = xgb_classifier.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.2f}')

Accuracy: 0.99


In [55]:
precision, recall, f1_score, _ = precision_recall_fscore_support(y_test, y_pred)

for i, v in enumerate(['p1', 'p2', 'p3', 'p4']):
    print(f"Class {v}:")
    print(f"Precision: {precision[i]}")
    print(f"Recall: {recall[i]}")
    print(f"F1 Score: {f1_score[i]}")
    print()

Class p1:
Precision: 0.9868421052631579
Recall: 0.9615384615384616
F1 Score: 0.974025974025974

Class p2:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

Class p3:
Precision: 0.9711324944485566
Recall: 0.990188679245283
F1 Score: 0.9805680119581465

Class p4:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0



In [56]:
#Hyperparameter tuning

from sklearn.model_selection import GridSearchCV
x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42)

In [57]:
## Define the XGBClassifier with the initial set of hyperparameters
xgb_model = xgb.XGBClassifier(objective='multi:softmax', num_class=4)

In [58]:
# Define the parameter grid for hyperparameter tuning

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
}

grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(x_train, y_train)

# Print the best hyperparameters
print("Best Hyperparameters:", grid_search.best_params_)

Best Hyperparameters: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 200}


In [59]:
# Evaluate the model with the best hyperparameters on the test set
best_model = grid_search.best_estimator_
accuracy = best_model.score(x_test, y_test)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.9945322714846072


In [60]:
# Define the hyperparameter grid
param_grid = {
  'colsample_bytree': [0.1, 0.3, 0.5, 0.7, 0.9],
  'learning_rate'   : [0.001, 0.01, 0.1, 1],
  'max_depth'       : [3, 5, 8, 10],
  'alpha'           : [1, 10, 100],
  'n_estimators'    : [10,50,100]
}

index = 0

answers_grid = {
    'combination'       :[],
    'train_Accuracy'    :[],
    'test_Accuracy'     :[],
    'colsample_bytree'  :[],
    'learning_rate'     :[],
    'max_depth'         :[],
    'alpha'             :[],
    'n_estimators'      :[]

    }

In [61]:
# Loop through each combination of hyperparameters
for colsample_bytree in param_grid['colsample_bytree']:
  for learning_rate in param_grid['learning_rate']:
    for max_depth in param_grid['max_depth']:
      for alpha in param_grid['alpha']:
          for n_estimators in param_grid['n_estimators']:

              index = index + 1

              # Define and train the XGBoost model
              model = xgb.XGBClassifier(objective='multi:softmax',
                                       num_class=4,
                                       colsample_bytree = colsample_bytree,
                                       learning_rate = learning_rate,
                                       max_depth = max_depth,
                                       alpha = alpha,
                                       n_estimators = n_estimators)



              y = df_encoded['Approved_Flag']
              x = df_encoded. drop ( ['Approved_Flag'], axis = 1 )

              label_encoder = LabelEncoder()
              y_encoded = label_encoder.fit_transform(y)


              x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42)


              model.fit(x_train, y_train)



              # Predict on training and testing sets
              y_pred_train = model.predict(x_train)
              y_pred_test = model.predict(x_test)


              # Calculate train and test results

              train_accuracy =  accuracy_score (y_train, y_pred_train)
              test_accuracy  =  accuracy_score (y_test , y_pred_test)



              # Include into the lists
              answers_grid ['combination']   .append(index)
              answers_grid ['train_Accuracy']    .append(train_accuracy)
              answers_grid ['test_Accuracy']     .append(test_accuracy)
              answers_grid ['colsample_bytree']   .append(colsample_bytree)
              answers_grid ['learning_rate']      .append(learning_rate)
              answers_grid ['max_depth']          .append(max_depth)
              answers_grid ['alpha']              .append(alpha)
              answers_grid ['n_estimators']       .append(n_estimators)


              # Print results for this combination
              print(f"Combination {index}")
              print(f"colsample_bytree: {colsample_bytree}, learning_rate: {learning_rate}, max_depth: {max_depth}, alpha: {alpha}, n_estimators: {n_estimators}")
              print(f"Train Accuracy: {train_accuracy:.2f}")
              print(f"Test Accuracy : {test_accuracy :.2f}")
              print("-" * 30)


Combination 1
colsample_bytree: 0.1, learning_rate: 0.001, max_depth: 3, alpha: 1, n_estimators: 10
Train Accuracy: 0.61
Test Accuracy : 0.60
------------------------------
Combination 2
colsample_bytree: 0.1, learning_rate: 0.001, max_depth: 3, alpha: 1, n_estimators: 50
Train Accuracy: 0.61
Test Accuracy : 0.60
------------------------------
Combination 3
colsample_bytree: 0.1, learning_rate: 0.001, max_depth: 3, alpha: 1, n_estimators: 100
Train Accuracy: 0.61
Test Accuracy : 0.60
------------------------------
Combination 4
colsample_bytree: 0.1, learning_rate: 0.001, max_depth: 3, alpha: 10, n_estimators: 10
Train Accuracy: 0.61
Test Accuracy : 0.60
------------------------------
Combination 5
colsample_bytree: 0.1, learning_rate: 0.001, max_depth: 3, alpha: 10, n_estimators: 50
Train Accuracy: 0.61
Test Accuracy : 0.60
------------------------------
Combination 6
colsample_bytree: 0.1, learning_rate: 0.001, max_depth: 3, alpha: 10, n_estimators: 100
Train Accuracy: 0.61
Test Accu

KeyboardInterrupt: 

In [63]:
#For unseen data

# # Your program code goes here
# start_time = time.time()


a3 = pd.read_excel("/content/Unseen_Dataset.xlsx")

In [64]:
cols_in_df = list(df.columns)
cols_in_df.pop(42)

'GL_Flag'

In [66]:
# Filter cols_in_df to only include columns that actually exist in the unseen dataset a3,
# and exclude 'Approved_Flag' since it is the target variable to be predicted.
available_cols = [col for col in cols_in_df if col in a3.columns and col != 'Approved_Flag']
df_unseen = a3[available_cols]


In [67]:
df_unseen['MARITALSTATUS'].unique()
df_unseen['EDUCATION'].unique()
df_unseen['GENDER'].unique()
df_unseen['last_prod_enq2'].unique()
df_unseen['first_prod_enq2'].unique()

array(['PL', 'ConsumerLoan', 'others', 'AL', 'HL', 'CC'], dtype=object)

In [68]:
# Ordinal feature -- EDUCATION
# SSC : 1
# 12TH : 2
# GRADUTATE : 3
# UNDER GRADUATE : 3
# POST-GRADUATE : 4
# OTHERS : 1
# PROFESSIONAL : 3

# Others has to be verified by the business end user

df_unseen.loc[df_unseen['EDUCATION'] == 'SSC',['EDUCATION']]            = 1
df_unseen.loc[df_unseen['EDUCATION'] == '12TH',['EDUCATION']]           = 2
df_unseen.loc[df_unseen['EDUCATION'] == 'GRADUATE',['EDUCATION']]       = 3
df_unseen.loc[df_unseen['EDUCATION'] == 'UNDER GRADUATE',['EDUCATION']] = 3
df_unseen.loc[df_unseen['EDUCATION'] == 'POST-GRADUATE',['EDUCATION']]  = 4
df_unseen.loc[df_unseen['EDUCATION'] == 'OTHERS',['EDUCATION']]         = 1
df_unseen.loc[df_unseen['EDUCATION'] == 'PROFESSIONAL',['EDUCATION']]   = 3

In [69]:
df_unseen['EDUCATION'].value_counts()
df_unseen['EDUCATION'] = df_unseen['EDUCATION'].astype(int)
df_unseen.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 41 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   pct_tl_open_L6M            100 non-null    float64
 1   pct_tl_closed_L6M          100 non-null    float64
 2   Tot_TL_closed_L12M         100 non-null    int64  
 3   pct_tl_closed_L12M         100 non-null    float64
 4   Tot_Missed_Pmnt            100 non-null    int64  
 5   CC_TL                      100 non-null    int64  
 6   Home_TL                    100 non-null    int64  
 7   PL_TL                      100 non-null    int64  
 8   Secured_TL                 100 non-null    int64  
 9   Unsecured_TL               100 non-null    int64  
 10  Other_TL                   100 non-null    int64  
 11  Age_Oldest_TL              100 non-null    int64  
 12  Age_Newest_TL              100 non-null    int64  
 13  time_since_recent_payment  100 non-null    int64  


/tmp/ipykernel_628/3409362196.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unseen['EDUCATION'] = df_unseen['EDUCATION'].astype(int)


In [70]:
df_encoded_unseen = pd.get_dummies(df_unseen, columns=['MARITALSTATUS','GENDER', 'last_prod_enq2' ,'first_prod_enq2'])


df_encoded_unseen.info()
k = df_encoded_unseen.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 53 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   pct_tl_open_L6M               100 non-null    float64
 1   pct_tl_closed_L6M             100 non-null    float64
 2   Tot_TL_closed_L12M            100 non-null    int64  
 3   pct_tl_closed_L12M            100 non-null    float64
 4   Tot_Missed_Pmnt               100 non-null    int64  
 5   CC_TL                         100 non-null    int64  
 6   Home_TL                       100 non-null    int64  
 7   PL_TL                         100 non-null    int64  
 8   Secured_TL                    100 non-null    int64  
 9   Unsecured_TL                  100 non-null    int64  
 10  Other_TL                      100 non-null    int64  
 11  Age_Oldest_TL                 100 non-null    int64  
 12  Age_Newest_TL                 100 non-null    int64  
 13  time_s

In [71]:
model = xgb.XGBClassifier(objective='multi:softmax',
                                     num_class=4,
                                     colsample_bytree=0.9,
                                     learning_rate=1,
                                     max_depth=3,
                                     alpha=10,
                                     n_estimators=100)

In [72]:
model.fit(x_train, y_train)


XGBClassifier(alpha=10, base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, ...)

In [74]:
# Align the columns of df_encoded_unseen with the training features 'x'
# This adds missing columns as 0/False and ensures the exact same column order
df_encoded_unseen_aligned = df_encoded_unseen.reindex(columns=x.columns, fill_value=0)

# Make predictions using the aligned dataset
y_pred_unseen = model.predict(df_encoded_unseen_aligned)


In [75]:
a3['Target_variable'] = y_pred_unseen